In [1]:
# Lab 4: Compare RNN vs LSTM vs GRU Performance

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Hyperparameters
input_size = 28
sequence_length = 28
num_layers = 2
hidden_size = 128
num_classes = 10
batch_size = 64
learning_rate = 0.001
epochs = 3

# Dataset
transform = transforms.ToTensor()

train_dataset = datasets.MNIST(root='./data', train=True, transform=transform, download=True)
test_dataset = datasets.MNIST(root='./data', train=False, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Base Model
class RNNModel(nn.Module):
    def __init__(self, model_type):
        super(RNNModel, self).__init__()

        self.model_type = model_type

        if model_type == "RNN":
            self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

        elif model_type == "LSTM":
            self.rnn = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)

        elif model_type == "GRU":
            self.rnn = nn.GRU(input_size, hidden_size, num_layers, batch_first=True)

        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        h0 = torch.zeros(num_layers, x.size(0), hidden_size).to(device)

        if self.model_type == "LSTM":
            c0 = torch.zeros(num_layers, x.size(0), hidden_size).to(device)
            out, _ = self.rnn(x, (h0, c0))
        else:
            out, _ = self.rnn(x, h0)

        out = self.fc(out[:, -1, :])
        return out

# Training Function
def train_model(model, name):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    print(f"\nTraining {name}")

    for epoch in range(epochs):
        model.train()

        for images, labels in train_loader:
            images = images.squeeze(1).to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")

    evaluate_model(model, name)

# Evaluation Function
def evaluate_model(model, name):
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.squeeze(1).to(device)
            labels = labels.to(device)

            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f"{name} Accuracy: {accuracy:.2f}%")

# Train Models
rnn_model = RNNModel("RNN").to(device)
lstm_model = RNNModel("LSTM").to(device)
gru_model = RNNModel("GRU").to(device)

train_model(rnn_model, "Simple RNN")
train_model(lstm_model, "LSTM")
train_model(gru_model, "GRU")

100%|██████████| 9.91M/9.91M [00:00<00:00, 18.2MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 495kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.59MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 3.73MB/s]



Training Simple RNN
Epoch [1/3], Loss: 0.2227
Epoch [2/3], Loss: 0.0718
Epoch [3/3], Loss: 0.1051
Simple RNN Accuracy: 95.05%

Training LSTM
Epoch [1/3], Loss: 0.0831
Epoch [2/3], Loss: 0.0111
Epoch [3/3], Loss: 0.0025
LSTM Accuracy: 97.86%

Training GRU
Epoch [1/3], Loss: 0.0802
Epoch [2/3], Loss: 0.0083
Epoch [3/3], Loss: 0.0159
GRU Accuracy: 98.17%
